# Filter LSST alerts to select high redshift objects

[Getting LSST alerts with the Babamul API](https://github.com/boom-astro/babamul/blob/main/examples/stream-basic/notebook.ipynb)

### Notes on running this notebook:

This notebook fetches alerts using the Babamul broker, which requires [user credentials saved in a .env file](https://github.com/boom-astro/babamul/blob/main/examples/stream-basic/.env.example). 

Babamul has periodically changed the naming schema, for example the token was previously changed from BABAMUL_API_TOKEN to BABAMUL_TOKEN, so the .env.example linked here may be incorrect.

In [ ]:
#imports 
import dotenv
dotenv.load_dotenv()

### Catalog uniformization

The following code needs to be edited to work for each specific catalog.

We cut the full catalog to subsection of confident galaxies

We rename columns with standardized schema (id, ra, dec, z, and any unique values that we want to keep for a given catalog)

In [ ]:
from astropy.table import Table                                                                                                                                             
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# we load the full downloaded catalog

with fits.open('../../data/catalogs/ASTRODEEP-JWST_photoz/ABELL2744_photoz.fits') as hdul:
    hdul.info()

t = Table.read('../../data/catalogs/ASTRODEEP-JWST_photoz/PRIMER-UDS_photoz.fits')                                                                                                 
t[0:5]

In [ ]:
# inspect catalog contents, ie histogram redshift values

import matplotlib.pyplot as plt

plot = plt.hist(t['zspec'], bins=50, range=(0,6))

In [ ]:
# values to select on are specific to the catalog

# here are our cuts for the clauds catalogs (https://www.clauds.net/available-data)
galaxies = t[
    (t['OBJ_TYPE'] == 0) &
    (t['CHI_BEST'] > 0) &        # has a valid SED fit
    (t['Z_BEST'] > 0) &          # has a valid photo-z
    (t['MASK'] == 0) &           # not in a bad region
    (t['CLASS_STAR_HSC_I'] < 0.9)  # morphologically not a star
]

print(f'selected {len(galaxies)} from {len(t)} total objects')


# and then here is where we standardize column names from the Clauds names
cut_catalog = galaxies['ID', 'RA', 'DEC', 'ZPHOT',
                        'Z_BEST68_LOW', 'Z_BEST68_HIGH', 'CHI_BEST',
                        'OBJ_TYPE', 'CLASS_STAR_HSC_I',
                        ].copy()

cut_catalog.rename_column('ID', 'id')
cut_catalog.rename_column('RA', 'ra')
cut_catalog.rename_column('DEC', 'dec')
cut_catalog.rename_column('ZPHOT', 'z')

In [ ]:
# here are our cuts for the astrodeep catalogs (http://www.astrodeep.eu/astrodeep-jwst-catalogs/)
# see Merlin 2024 for description of data, flags etc (https://arxiv.org/abs/2409.00169)

galaxies = t[
    (t['flag'] < 100000) # spurious sources
]

print(f'selected {len(galaxies)} from {len(t)} total objects')

# and then here is where we standardize column names from the astrodeep names
# we dont cut any columns as astrodeep is already very concise.

cut_catalog = galaxies.copy()
cut_catalog.rename_column('ID', 'id')
cut_catalog.rename_column('RA', 'ra')
cut_catalog.rename_column('DEC', 'dec')
cut_catalog['z'] = np.where(cut_catalog['zspec'] != -99.0, cut_catalog['zspec'], cut_catalog['zphot'])                                                            

In [ ]:
# and then we print some statistics on the selected objects, which are copied into catalogs_catalog.py

for col in ['ra', 'dec', 'z']:                                                                                                                                         
      print(f"{col}: min={cut_catalog[col].min():.4f}, max={cut_catalog[col].max():.4f}") 

print(f'median z: {np.median(cut_catalog['z']):.4f}')

# finally we save the cut catalog to a fits file
cut_catalog.write('../../data/catalogs/ASTRODEEP_PRIMERUDS_cut.fits', overwrite=False) 

In [ ]:
from thor.utils.rubin_stats_visualizations import plot_skymap

plot_skymap(
    alerts=None,
    plot_catalogs=True,
    plot_ddf=False,
    catalog_path="../../data/catalogs",
    title="Catalog Coverage Skymap",
)

### Fetching, prefiltering, organizing LSST alerts

Develop pipeline to crossmatch (or do any inference) on sources locally

In [ ]:
# get alerts from date range and other basic filters for astrophysical sources
# this has crashed for large date ranges, so is built to save alerts in batches.

from thor.utils.fetch_alerts import babamul_get_alerts

start="07-01-2026"
end="07-02-2026"

loaded_alerts = babamul_get_alerts(
    survey="LSST",
    start_time=start,
    end_time=end,
    min_drb=0.4,
    is_rock=False,
    is_star=False,
    is_near_brightstar=False,
    is_stationary=True,
) 

In [ ]:
# Bookeeping: if we saved batches as we fetched, combine the batched files of alerts into a single compressed file and move out of raw_files
from thor.utils.fetch_alerts import combine_alert_files
from datetime import datetime

start_fmt = datetime.strptime(start, "%m-%d-%Y").strftime("%m%d%y")                                                                                                         
end_fmt = datetime.strptime(end, "%m-%d-%Y").strftime("%m%d%y")
filename = f"basic_{start_fmt}_{end_fmt}.json.gz" 

combined = combine_alert_files(input_dir="../../data/lsst_alert_download/raw_files/", output_path=f"../../data/lsst_alert_download/{filename}", delete_raw=True)

In [ ]:
# Bookkeeping: combine saved alerts to bigger batches per file
# this will deduplicated using objectId when running

from thor.utils.fetch_alerts import combine_alert_files

combined = combine_alert_files(                                                                                                                                           
    input_dir="../../data/lsst_alert_download/",                                                                                                                      
    input_files=["basic_041226_062926.json.gz", "basic_062626_070226.json.gz"],
    output_path="../../data/lsst_alert_download/basic_041226_070226.json.gz",
    delete_raw=True                                                                                                         
)

In [ ]:
# if we want to directly open a locally saved alerts file, we can do that too

from thor.utils.fetch_alerts import load_alerts
loaded_alerts = load_alerts("../../data/lsst_alert_download/basic_010126_041426.json.gz")

In [ ]:
# additional filtering

# in generic_filter we make some more generic cuts to get real astrophysical objects
# filter out dipole (image subtraction artifacts), poor PSF fits, low SNR, extended sources, and alerts with shape or centroid fitting issues. 
# filter currently repeats cuts made in babamul_get_alerts: drb<0.4, rock, star, near_brightstar, isdiffpos, and stationary.

from thor.utils.filter_functions import (filter_alerts, generic_filter, deduplicate_alerts)

filtered_alerts = filter_alerts(                                                                                                           
    loaded_alerts,                                                            
    generic_filter,                                                                                                                        
) 

# deduplicate alerts to get unique objects
filtered_objects = deduplicate_alerts(filtered_alerts)

### Catalog crossmatch

In [ ]:
# crossmatch with all available catalogs, ie all .fits files in data/catalogs
# returns a dictionary with all matched alert and catalog info.
# can provide catalog_name as a string or list of strings to specify which catalogs in catalog_path (default data/catalogs) to use. 
# user must download catalogs and ensure there are columns with names id, ra, dec, and z - the function is not flexible in 
# identifying different naming schema for these columns.

from thor.utils.filter_functions import catalog_crossmatch

crossmatched_objects = catalog_crossmatch(
      alerts=filtered_objects,
)

In [ ]:
# additional filtering on catalog features

from thor.utils.filter_functions import catalog_filter

filtered = catalog_filter(
    crossmatched_objects,
    z_min = 0.5,
)

In [ ]:
# Source specific filtering:
# in tde_filter we also make more specific cuts, including minimum # detections, no milliquas matches

from thor.utils.filter_functions import (filter_alerts, tde_filter)

filtered = filter_alerts(                                                                                                           
    filtered,                                                            
    tde_filter
) 

### loading and inspecting objects

In [ ]:
crossmatched_alerts = [v["LSST"] for v in crossmatched_objects.values()]
print(f"Total crossmatched alerts: {len(crossmatched_alerts)}")

filtered_crossmatched_alerts = [v["LSST"] for v in filtered.values()]  
print(f"Filtered crossmatched alerts: {len(filtered_crossmatched_alerts)}")

In [ ]:
# inspect all crossmatch results without the cuts on rise etc.

from babamul.jupyter import scan_alerts
from astropy.time import Time
print(f'Todays JD: {round(Time.now().jd)}')
scan_alerts(filtered_crossmatched_alerts)

In [ ]:
# copy lsst id to print crossmatch info for a given id from dictionary

id = '313967767357227074'

obj = crossmatched_objects[id]
for catalog, data in obj.items():
     if catalog != "LSST" and data is not None:
         print(f"{catalog}:")
         print(f"end=z = {data['z']:.2f}") 
         if "CHI_BEST" in data:
            print(f"chi2 = {data['CHI_BEST']:.2f}")
         if "consearch_arcsecs" in data:
            print(f"separation = {data['consearch_arcsecs']:.2f} arcsec")


In [ ]:
# check abs mag given apparent mag and redshift
m=22
z=2.1

from astropy.cosmology import Planck18 as cosmo                                                                                                                   
abs_mag = m - cosmo.distmod(z).value
print(f"Apparent magnitude: {m}, redshift: {z}, absolute magnitude: {abs_mag:.2f}")

In [ ]:
# by hand create a list of objects that pass inspection, save
# will add objectid to existing file, without duplicating objects

cand_ids_to_save = ['313853517979713615', '170050498847047681', '170389126953566274', '170270398108139547', '313980947917701210',
                    '170591539645907039', '170591539376422992', '170054972075409453', '170046090194714651', '170032910440071196',
                    '313853533015244928', '313998569368453223', '313967767357227074']

from thor.utils.fetch_alerts import save_objects

save_objects(cand_ids_to_save, path="../../data/lsst_alert_download/tde_candidates.json.gz")

### additional ways to fetch objects

In [ ]:
# or we provide function to run cosmos crossmatch by directly provided ra and dec instead of alert
# you may provide a single ra and dec, or a list of ras and a list of corresponding decs as arguments.

from thor.utils.filter_functions import catalog_crossmatch

crossmatched_coords = catalog_crossmatch(
    ra=150.101273,
    dec=2.743894,
    radius_arcsec=300,
    catalog_path="../../data/catalogs",
)

In [ ]:
# or we can load our list of previously saved candidates, and fetch alerts to display

from thor.utils.filter_functions import catalog_crossmatch
from thor.utils.fetch_alerts import (load_objects, fetch_latest_alerts)

object_ids = load_objects("../../data/lsst_alert_download/tde_candidates.json.gz")
latest_alerts = fetch_latest_alerts(object_ids, save=False, verbose=False)
saved_objects = catalog_crossmatch(latest_alerts)

In [ ]:
# look at our candidates:

from babamul.jupyter import scan_alerts
from astropy.time import Time
print(f'Todays JD: {round(Time.now().jd)}')

show_saved_objects = [v["LSST"] for v in saved_objects.values()]  
scan_alerts(show_saved_objects)

In [ ]:
# copy lsst id to print crossmatch info for a given id from dictionary

id = '170591539376422992'

obj = saved_objects[id]
for catalog, data in obj.items():
     if catalog != "LSST" and data is not None:
         print(f"{catalog}:")
         print(f"end=z = {data['z']:.2f}") 
         if "CHI_BEST" in data:
            print(f"chi2 = {data['CHI_BEST']:.2f}")
         if "consearch_arcsecs" in data:
            print(f"separation = {data['consearch_arcsecs']:.2f} arcsec")


In [ ]:
obj

### prost - currently developing bayesian crossmatch instead of basic conesearch

https://github.com/alexandergagliano/Prost

notes: set priors on z, offset, host mag.

missed catalog probability: true host too faint and undetected in image

small cone probability: search radius too small

can return top 2 best host associations.

In [ ]:
import importlib
import thor.utils.fetch_alerts
importlib.reload(thor.utils.fetch_alerts)

In [ ]:
# FIXME: make normal conesearch on relatively large radius (eg 15"), return all matches
# for matches, run prost to select match w/ probability

In [ ]:
# test Prost out of the box / basic usage

import pandas as pd
from astro_prost import associate_sample
from scipy.stats import gamma, halfnorm, uniform

# define a transient catalog 
transient_catalog = pd.DataFrame({
    'name': ['MyTransient'],
    'ra': [350.838161],
    'dec': [0.101394]
})

# define a set of catalogs to search -- options are glade, decals, panstarrs, and skymapper
catalogs = ["decals", "glade"]

# define priors and likelihoods
priorfunc_offset = uniform(loc=0, scale=10)
likefunc_offset = gamma(a=0.75)

priors = {"offset": priorfunc_offset}
likes = {"offset": likefunc_offset}

# associate
hosts = \
    associate_sample(
        transient_catalog,
        priors=priors,
        likes=likes,
        catalogs=catalogs,
        name_col='name',
        coord_cols=('ra', 'dec'),
        save=False
)

In [ ]:
# test prost with custom patch to handle unsuported catalogs

from pathlib import Path    
import pandas as pd                                                                                                                                                
from astropy.table import Table                                                                                                                                             
from thor.utils.prost_catalogs import register_local_catalog
from astro_prost import associate_sample
from scipy.stats import gamma, halfnorm, uniform



# custom local catalogs                                                                                                              
CATALOG_DIR = Path.cwd().parents[1] / "data" / "catalogs"
df = Table.read(CATALOG_DIR / "COSMOS2025_cut.fits").to_pandas()                                                                                                            
register_local_catalog("cosmos2025", df)

# define a transient catalog 
transient_catalog = pd.DataFrame({
    'name': ['MyTransient'],
    'ra': [150.1],
    'dec': [2.2]
})

# define a set of catalogs to search -- options are glade, decals, panstarrs, and skymapper
catalogs = ["cosmos2025"]

# define priors and likelihoods
priorfunc_offset = uniform(loc=0, scale=10)
likefunc_offset = gamma(a=0.75)
priors = {"offset": priorfunc_offset}
likes = {"offset": likefunc_offset}

# associate
hosts = \
    associate_sample(
        transient_catalog,
        priors=priors,
        likes=likes,
        catalogs=catalogs,
        name_col='name',
        coord_cols=('ra', 'dec'),
        save=False
)